In [0]:
import hashlib
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = "workspace"
SCHEMA = "ecom_poc"
RAW_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/raw_files"

ORDERS_SCHEMA = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("country", StringType(), True),
])

TXNS_SCHEMA = StructType([
    StructField("txn_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("txn_status", StringType(), True),
    StructField("txn_ts", StringType(), True),
])

def get_deterministic_batch_id(file_path: str) -> str:
    files = dbutils.fs.ls(file_path)
    meta = f"{files[0].path}|{files[0].size}|{files[0].modificationTime}"
    return hashlib.sha256(meta.encode("utf-8")).hexdigest()[:16]

def ingest_bronze_csv(file_name: str, explicit_schema: StructType, target_table: str):
    full_path = f"{RAW_BASE}/{file_name}"
    full_table_name = f"{CATALOG}.{SCHEMA}.{target_table}"
    batch_id = get_deterministic_batch_id(full_path)
    data_cols = [field.name for field in explicit_schema.fields]

    raw_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .schema(explicit_schema)
        .csv(full_path)
    )

    row_hash_expr = F.sha2(
        F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("NULL")) for c in data_cols]),
        256
    )

    bronze_df = (
        raw_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_row_hash", row_hash_expr)
        .withColumn(
            "_bronze_record_id",
            F.sha2(F.concat_ws("||", F.lit(batch_id), F.col("_row_hash"), F.monotonically_increasing_id().cast("string")), 256)
        )
    )

    # Idempotent load: remove existing rows from this exact batch before appending
    if spark.catalog.tableExists(full_table_name):
        spark.sql(f"DELETE FROM {full_table_name} WHERE _batch_id = '{batch_id}'")

    bronze_df.write.format("delta").mode("append").saveAsTable(full_table_name)
    print(f"Loaded {full_table_name} with batch_id={batch_id}")

# Run ingestion twice on orders to prove idempotency (no double-counting)
ingest_bronze_csv("orders.csv", ORDERS_SCHEMA, "bronze_orders")
ingest_bronze_csv("orders.csv", ORDERS_SCHEMA, "bronze_orders")
ingest_bronze_csv("transactions.csv", TXNS_SCHEMA, "bronze_transactions")

In [0]:
%sql
-- 1. Verify exact 1,000,000 lossless row count & idempotency
SELECT 'bronze_orders' AS table_name, _batch_id, COUNT(*) AS row_count
FROM workspace.ecom_poc.bronze_orders GROUP BY _batch_id
UNION ALL
SELECT 'bronze_transactions' AS table_name, _batch_id, COUNT(*) AS row_count
FROM workspace.ecom_poc.bronze_transactions GROUP BY _batch_id;


In [0]:
%sql
-- 2. Inspect schema (all source fields are STRING)
DESCRIBE TABLE workspace.ecom_poc.bronze_orders;

In [0]:
%sql
-- 3. Sample rows with audit columns
SELECT * FROM workspace.ecom_poc.bronze_orders LIMIT 5;